# Minilink showcase

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/showcase_minilink.ipynb)

**Write the equations once. Simulate, analyze, control, plan, optimize, learn.**

A minilink model is one equation, the state `x`, the input `u`, the time `t` and the
parameters `p`:

$$\dot x = f(x, u, t;\, p)$$

This notebook writes that equation once, for a pendulum, and hands the same object to
every tool in the library:

1. [Write the equation](#1.-Write-the-equation)
2. [Close a loop](#2.-Close-a-loop)
3. [Linearize, analyze, design](#3.-Linearize,-analyze,-design)
4. [State a task once, plan it three ways](#4.-State-a-task-once,-plan-it-three-ways)
5. [Put a planner in the loop](#5.-Put-a-planner-in-the-loop)
6. [Learn a law, then analyze it like any other](#6.-Learn-a-law,-then-analyze-it-like-any-other)
7. [Differentiate through everything](#7.-Differentiate-through-everything)
8. [The same lines on a six-axis robot](#8.-The-same-lines-on-a-six-axis-robot)

It runs in about a minute. The tutorials (`basics/`, `core/`, `advanced/`) take each step
further.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. Write the equation

A pendulum driven by a torque $\tau$, its angle $\theta$ measured from the hanging
position:

$$m l^2\, \ddot\theta = \tau - m g l \sin\theta - d\, \dot\theta$$

The class holds its numbers in `params` and writes `f` as the textbook does. `tf` places
a frame on the rod, and the skin draws on it: that is all the animation needs.

In [ ]:
import numpy as np

from minilink import DynamicSystem
from minilink.core.backends import array_module
from minilink.core.kinematics import SE2
from minilink.graphical.animation.primitives import Circle, Rod


class MyPendulum(DynamicSystem):
    def __init__(self):
        super().__init__(state_dim=2, input_dim=1, output_dim=2, expose_state=True)

        # Parameters: mass [kg], length [m], gravity [m/s^2], damping [N m s]
        self.params = {"m": 1.0, "l": 1.0, "g": 9.81, "d": 0.1}

        # The torque the motor has, and the states a task may visit
        self.inputs["u"].lower = np.array([-5.0])
        self.inputs["u"].upper = np.array([5.0])
        self.state.lower = np.array([-2.0 * np.pi, -8.0])
        self.state.upper = np.array([2.0 * np.pi, 8.0])
        self.state.labels = ["theta", "dtheta"]

        # Graphics: a rod hanging from the pivot and a bob at its end
        self.skin = lambda sys: {
            "rod": [Rod(length=1.0, radius=0.03), Circle(radius=0.08, center=[0.0, -1.0], fill=True)]
        }

    def f(self, x, u, t=0.0, params=None):
        params = self.params if params is None else params
        xp = array_module(x)

        m = params["m"]
        l = params["l"]
        g = params["g"]
        d = params["d"]

        theta = x[0]
        dtheta = x[1]
        tau = u[0]

        # (m l²) θ̈ = τ − m g l sin θ − d θ̇
        ddtheta = (tau - m * g * l * xp.sin(theta) - d * dtheta) / (m * l**2)
        dx = xp.array([dtheta, ddtheta])

        return dx

    def tf(self, x, u, t=0.0, params=None):
        return {"rod": SE2(0.0, 0.0, x[0])}


plant = MyPendulum()
plant.x0 = np.array([2.0, 0.0])  # released at 2 rad, at rest
plant.compute_trajectory(tf=10.0)
plant.plot_trajectory()

In [ ]:
plant.animate()

`plot_phase_plane` draws the vector field of `f` with the simulated path on top: one
picture of what the equation does.

In [ ]:
plant.plot_phase_plane()

## 2. Close a loop

`controller @ plant` closes the feedback loop. `Impedance` is a spring and a damper on
the joint, $\tau = K_p (r - \theta) - K_d\, \dot\theta$; a `Step` source moves its
reference to 1 rad. The loop is a system like the plant: the same calls draw it,
simulate it and plot it.

In [ ]:
from minilink import Impedance, Step, closed_loop

controller = Impedance(Kp=30.0, Kd=8.0)
loop = closed_loop(controller, plant, reference=Step(final_value=[1.0], step_time=2.0))
loop.plot_diagram()

In [ ]:
plant.x0 = np.zeros(2)
loop.compute_trajectory(tf=8.0)
loop.plot_trajectory()

`%` puts a block on a clock: the same law run by a computer at 10 Hz, on the
continuous plant.

In [ ]:
digital_loop = (controller % 0.1) @ plant
digital_loop.compute_trajectory(tf=8.0)
digital_loop.plot_trajectory()

## 3. Linearize, analyze, design

The upright position, $\theta = \pi$, is an equilibrium. `linearize` gives the local
model there, and its poles say it is unstable. `lqr` designs a state feedback on that
model and returns a controller block; the closed loop is linearized the same way.

In [ ]:
from minilink import lqr

x_up = np.array([np.pi, 0.0])
balance = lqr(plant, Q=np.eye(2), R=np.eye(1), x_bar=x_up) @ plant

print("poles of the plant, upright:", np.linalg.eigvals(plant.linearize(x_up).A()))
print("poles of the LQR loop      :", np.linalg.eigvals(balance.linearize(x_up).A()))

In [ ]:
plant.x0 = np.array([np.pi - 0.5, 0.0])  # 0.5 rad from the upright
balance.compute_trajectory(tf=5.0)
balance.plot_trajectory()

The frequency response of the same model, hanging down, from the torque to the angle:

In [ ]:
plant.x0 = np.zeros(2)
plant.plot_bode()

## 4. State a task once, plan it three ways

Swing the pendulum from rest to the upright with a motor too weak to lift it directly.
A `PlanningProblem` states the task, and nothing about how to solve it: the system, a
cost, the allowed states and torques, the start and the goal.

In [ ]:
from minilink import BallSet, PlanningProblem, QuadraticCost

cost = QuadraticCost.from_system(plant, x_bar=x_up, Q=np.eye(2), R=np.eye(1))
problem = PlanningProblem(
    plant,
    x0=np.zeros(2),
    cost=cost,
    X=plant.state.box,
    U=plant.input_box,
    Xf=BallSet(x_up, 0.2),
    horizon=4.0,
)
problem

**Value iteration** grids the state space and solves the Bellman equation. Its answer
is a feedback law over the whole grid, and the cost-to-go $J(x)$ from every state.

In [ ]:
from minilink import DynamicProgrammingPlanner

vi = DynamicProgrammingPlanner(
    problem.replace(horizon=None),  # the converged law: no horizon
    x_grid=(101, 101),
    u_grid=(11,),
    dt=0.05,
    tol=0.5,
    out_of_bound_cost=500.0,
)
vi.solve()
vi.plot_cost_to_go(jmax=500.0)

**RRT** grows a tree of motions the pendulum can make, until a branch reaches the
goal.

In [ ]:
from minilink import RRTPlanner

rrt = RRTPlanner(problem, seed=0, max_iterations=5000)
rrt_solution = rrt.solve()
rrt.plot_tree(x_axis=0, y_axis=1);

**Trajectory optimization** transcribes the problem into a nonlinear program over 40
time steps and solves it.

In [ ]:
from minilink import TrajectoryOptimizationPlanner

trajopt = TrajectoryOptimizationPlanner(problem, n_steps=40, transcription="direct_collocation")
trajopt_solution = trajopt.solve()
print(trajopt_solution)

Three answers to one problem, on one phase plane: the optimized trajectory, with the
two others drawn over it. Value iteration's law is a controller block:
`vi.controller @ plant` closes the loop like any other.

In [ ]:
import matplotlib.pyplot as plt

vi_trajectory = (vi.controller @ plant).compute_trajectory(tf=10.0)

phase = plant.plot_phase_plane(traj=trajopt_solution.trajectory, show=False)
phase.axes.plot(*vi_trajectory.x, "C0--", label="value iteration")
phase.axes.plot(*rrt_solution.trajectory.x, "C2", label="RRT")
phase.axes.legend(loc="lower right")
plt.show()

## 5. Put a planner in the loop

A plan computed once is open loop. `ModelPredictiveController` wraps a planner and
solves it again at every tick from the measured state; on a 10 Hz clock it is a digital
controller, closed on the plant with the same `@`.

In [ ]:
from minilink.control import ModelPredictiveController

horizon_cost = QuadraticCost.from_system(
    plant, x_bar=x_up, Q=np.eye(2), R=0.1 * np.eye(1), S=10.0 * np.eye(2)
)
horizon_problem = PlanningProblem(
    plant, x0=np.zeros(2), cost=horizon_cost, U=plant.input_box, horizon=3.0
)
horizon_planner = TrajectoryOptimizationPlanner(
    horizon_problem, n_steps=30, backend="jax", method="slsqp", options={"maxiter": 100}
)

mpc = ModelPredictiveController(horizon_planner, delay=True)
mpc_loop = (mpc % 0.1) @ plant
mpc_trajectory = mpc_loop.compute_trajectory(tf=6.0)
plant.plot_trajectory(mpc_trajectory.of("plant"))

In [ ]:
plant.animate(mpc_trajectory.of("plant"))

## 6. Learn a law, then analyze it like any other

The same task for a learner: the start is now random, drawn from a `Uniform` law, and
the cost is zero at the upright. `ReinforcementLearningPlanner` trains a small network
on it with PPO.

In [ ]:
import jax.numpy as jnp

from minilink import CostFunction, ReinforcementLearningPlanner, Uniform


class SwingUpCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        theta = x[0]
        dtheta = x[1]
        tau = u[0]

        # 1 + cos θ is zero at the upright; a little speed and a little torque
        g = 1.0 + jnp.cos(theta) + 0.01 * dtheta**2 + 0.01 * tau**2

        return g

    def h(self, x, t=0.0, params=None):
        return 0.0


random_start = PlanningProblem(
    plant,
    cost=SwingUpCost(),
    horizon=np.inf,
    x0=Uniform([-np.pi, -1.0], [np.pi, 1.0]),
    U=plant.input_box,
)
learner = ReinforcementLearningPlanner(
    random_start,
    dt=0.05,
    features=lambda x: jnp.array([jnp.cos(x[0]), jnp.sin(x[0]), 0.1 * x[1]]),
    hidden=(32, 32),
    n_rollouts=64,
    n_steps_per_update=32,
    batch_size=256,
    learning_rate=3e-3,
    gamma=0.97,
)
learner.solve(n_timesteps=120_000)
learner.plot_learning_curve();

The trained network is a controller block. It closes the loop with `@`, and its law
is drawn over the state space.

In [ ]:
neural_law = learner.controller
neural_loop = neural_law @ plant

plant.x0 = np.array([0.05, 0.0])
neural_loop.compute_trajectory(tf=10.0, dt=0.01)
neural_loop.plot_trajectory()

In [ ]:
neural_law.plot_control_law(x_axis=0, y_axis=1, u_axis=0)

One judge for every answer: `evaluate` runs a controller on the problem's own random
starts. The learned law against the LQR of section 3, on the same 100 draws; the LQR
ignores the torque limit, and the count of feasible runs shows it.

In [ ]:
from minilink import evaluate

lqr_law = balance.subsystems["controller"]
runs = dict(dt=0.05, n_trials=100, episode_length=10.0, key=1)

print("neural law:", evaluate(random_start, neural_law, **runs))
print("LQR       :", evaluate(random_start, lqr_law, **runs))

A loop that holds a neural network is still one `f`, so it is linearized like the
others. Its poles at the upright:

In [ ]:
np.linalg.eigvals(neural_loop.linearize(x_up).A())

## 7. Differentiate through everything

`f` is a pure function, so on the JAX backend every quantity above has an exact
derivative. The sensitivity of the dynamics to each physical parameter:

In [ ]:
plant.jacobian("f", "params", np.array([0.5, 1.0]), np.zeros(1))

The gradient goes through a whole closed-loop simulation. The problem is compiled
with a controller into one function of the loop's parameters; an `Optimizer` then tunes
the two gains of the impedance law of section 2 on the cost of a 5 s step response.

In [ ]:
from minilink import Optimizer

step_cost = QuadraticCost.from_system(
    plant, Q=np.diag([1.0, 0.0]), R=0.01 * np.eye(1), x_bar=np.array([1.0, 0.0])
)
step_task = PlanningProblem(plant, x0=np.zeros(2), cost=step_cost, horizon=5.0)

evaluator = step_task.compile(Impedance(Kp=1.0, Kd=1.0), backend="jax")
start = evaluator.params
start["r"] = np.array([1.0])  # the reference: 1 rad

gains = ["controller.Kp", "controller.Kd"]
best = Optimizer(evaluator.program(gains, params=start), method="auto").solve()
tuned = evaluator.unpack(best.z, gains, params=start)

print("before:", evaluator.evaluate(start))
print("after :", evaluator.evaluate(tuned), " Kp, Kd =", best.z.round(2))
evaluator.loop.plot_trajectory(evaluator.trajectory(tuned))

The compiled model also runs in batches: a thousand pendulums, a thousand steps
each, in one call, each with its own rod length.

In [ ]:
compiled = plant.compile(backend="jax")

lengths = np.linspace(0.5, 2.0, 1000)
family = dict(plant.params, l=lengths, d=0.0)
starts = np.tile([1.0, 0.0], (1000, 1))
swings = np.asarray(compiled.rollout_batch(starts, n_steps=1000, dt=0.005, params=family))

t = 0.005 * np.arange(1001)
for k in range(0, 1000, 200):
    plt.plot(t, swings[k, :, 0], label=f"l = {lengths[k]:.2f} m")
plt.xlabel("t [s]")
plt.ylabel("theta [rad]")
plt.legend()
plt.show()

## 8. The same lines on a six-axis robot

Nothing above is special to the pendulum. A UR5 arm from the catalog, with its full
rigid-body dynamics, is the same kind of object: a law, `@`, a simulation, an animation.

In [ ]:
from minilink import JointImpedance, UR5Manipulator

arm = UR5Manipulator()
q_start = np.array([0.0, -np.pi / 2, 0.0, -np.pi / 2, 0.0, 0.0])
q_goal = np.array([1.35, -1.2, 1.55, -1.4, 0.35, 0.25])
arm.x0 = arm.q2x(q_start, np.zeros(6))

arm_law = JointImpedance(
    arm,
    gravity_comp=True,
    Kp=[10.0, 60.0, 20.0, 0.5, 0.1, 0.003],
    Kd=[3.0, 25.0, 8.0, 0.2, 0.05, 0.003],
)
arm_loop = Step(initial_value=q_start, final_value=q_goal, step_time=1.0) >> arm_law @ arm
arm_loop.compute_trajectory(tf=6.0, n_samples=180, backend="jax")
arm_loop.plot_trajectory()

In [ ]:
arm_loop.animate(renderer="meshcat", is_3d=True)

## One equation, every tool

| What was done | With |
| --- | --- |
| A model, simulated and animated | `DynamicSystem`, `compute_trajectory`, `animate` |
| A loop, continuous then on a clock | `@`, `closed_loop`, `%` |
| Poles, a Bode plot, a state feedback | `linearize`, `plot_bode`, `lqr` |
| A task, planned three ways | `PlanningProblem`; `DynamicProgrammingPlanner`, `RRTPlanner`, `TrajectoryOptimizationPlanner` |
| A planner in the loop | `ModelPredictiveController` |
| A learned law, scored and linearized | `ReinforcementLearningPlanner`, `evaluate`, `linearize` |
| Gains tuned through the simulation | `compile`, `Optimizer` |
| A thousand rollouts in one call | `rollout_batch` |

Every object in that table was built from the one `f` of section 1.